<a href="https://colab.research.google.com/github/DaminikaDz/Data_Ranking_Training_Dynamics/blob/main/src/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!wget https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz
!tar -xzf imagenette2-160.tgz

--2026-04-24 14:21:36--  https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.72.190, 52.216.49.32, 16.15.228.110, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.72.190|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 99003388 (94M) [application/x-tar]
Saving to: ‘imagenette2-160.tgz.1’

imagenette2-160.tgz 100%[===================>]  94.42M  16.1MB/s    in 7.4s    

2026-04-24 14:21:45 (12.7 MB/s) - ‘imagenette2-160.tgz.1’ saved [99003388/99003388]



In [13]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [14]:
!pip install torch torchvision tqdm scikit-learn pandas pillow -q

In [15]:
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14")
model = model.to(device)
model.eval()

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-23): 24 x NestedTensorBlock(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((1024,), eps=1e-06, element

In [21]:
from torchvision import datasets, transforms

In [22]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [23]:
train_ds = datasets.ImageFolder("imagenette2-160/train", transform=transform)
val_ds = datasets.ImageFolder("imagenette2-160/val", transform=transform)

print(len(train_ds), len(val_ds))

9469 3925


In [24]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [25]:
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F

def extract_embeddings(loader, model, device):
    all_embeddings = []
    all_labels = []
    all_paths = []

    with torch.inference_mode():
        for images, labels in tqdm(loader):
            images = images.to(device, non_blocking=True)

            emb = model(images)
            emb = F.normalize(emb, dim=1)

            all_embeddings.append(emb.cpu().numpy())
            all_labels.extend(labels.numpy())

    embeddings = np.vstack(all_embeddings)
    labels = np.array(all_labels)

    return embeddings, labels

In [26]:
train_embeddings, train_labels = extract_embeddings(train_loader, model, device)
val_embeddings, val_labels = extract_embeddings(val_loader, model, device)

print(train_embeddings.shape)
print(val_embeddings.shape)

100%|██████████| 62/62 [03:20<00:00,  3.23s/it]

(9469, 1024)
(3925, 1024)


In [27]:
np.save("train_embeddings.npy", train_embeddings)
np.save("val_embeddings.npy", val_embeddings)

np.save("train_labels.npy", train_labels)
np.save("val_labels.npy", val_labels)